# End-to-End RAG Pipeline

This notebook demonstrates a complete Retrieval-Augmented Generation (RAG) workflow using docling-pipelines.

## What You'll Learn

1. Build a small, deterministic corpus for RAG
2. Run a single docling-pipelines indexing flow: ingest → extract → chunk → embeddings → vector store
3. Inspect intermediate outputs after each stage
4. Inspect the OpenSearch index created by the flow
5. Retrieve relevant chunks for fixed user questions
6. Generate grounded answers with Ollama using retrieved evidence
7. Evaluate whether the generated answers match expected answers

## Corpus Used in This Notebook

This notebook intentionally indexes exactly three support documents from [`tests/fixtures/customer_support_docs/`](tests/fixtures/customer_support_docs):

- [`refund.txt`](tests/fixtures/customer_support_docs/refund.txt)
- [`account_cancel.txt`](tests/fixtures/customer_support_docs/account_cancel.txt)
- [`tech_support.txt`](tests/fixtures/customer_support_docs/tech_support.txt)

These files were chosen because they are short, readable, and contain explicit answers that make retrieval quality easy to inspect.

## Prerequisites

- Ollama running on `http://localhost:11434`
- Model `nomic-embed-text` pulled for embeddings: `ollama pull nomic-embed-text`
- Model `llama3.2` pulled for answer generation: `ollama pull llama3.2`
- OpenSearch running on `https://localhost:9200` (set `OPENSEARCH_USE_SSL=false` if using plain HTTP)
- Virtual environment activated

## Setup and Imports

In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

import pandas as pd
import requests

# Add src to path if needed
if 'PYTHONPATH' not in os.environ:
    src_path = Path.cwd().parent.parent / "src"
    sys.path.insert(0, str(src_path))

from docpipe.lib.docpipe_flow_manager import DocpipeFlowManager

print("Imports loaded successfully")

## Load Environment Variables

In [ ]:
# Set OpenSearch credentials
project_root = Path.cwd()
if (project_root / 'examples' / 'notebooks').exists():
    env_file = project_root / '.env'
else:
    env_file = project_root.parent.parent / '.env'

if env_file.exists():
    from dotenv import load_dotenv
    load_dotenv(env_file, override=True)
    print(f"Loaded credentials from .env file: {env_file}")
else:
    print(f".env file not found at: {env_file}")
    print("Tip: Copy .env.example to .env in project root and update OPENSEARCH_USERNAME and OPENSEARCH_PASSWORD")

# Optionally set environment variables manually in the notebook if needed
if not os.getenv('OPENSEARCH_USERNAME'):
    os.environ['OPENSEARCH_USERNAME'] = 'admin'
if not os.getenv('OPENSEARCH_PASSWORD'):
    os.environ['OPENSEARCH_PASSWORD'] = '<YOUR-OPENSEARCH-PASSWORD>'

# Set OPENSEARCH_USE_SSL=false in your environment if your OpenSearch uses plain HTTP
OPENSEARCH_USE_SSL = os.getenv('OPENSEARCH_USE_SSL', 'true').lower() != 'false'

print(f"OpenSearch credentials configured (username: {os.environ.get('OPENSEARCH_USERNAME', 'admin')})")
print(f"OpenSearch SSL: {OPENSEARCH_USE_SSL}")

## Notebook Configuration

In [ ]:
CONFIG = {
    "documents": [
        "../../tests/fixtures/customer_support_docs/refund.txt",
        "../../tests/fixtures/customer_support_docs/account_cancel.txt",
        "../../tests/fixtures/customer_support_docs/tech_support.txt",
    ],
    "chunk_size": 512,
    "chunk_overlap": 128,
    "embedding_model": "ollama/nomic-embed-text",
    "embedding_dimension": 768,
    "llm_model": "llama3.2",
    "ollama_base": "http://localhost:11434",
    "opensearch_host": "localhost",
    "opensearch_port": 9200,
    "opensearch_use_ssl": OPENSEARCH_USE_SSL,
    "opensearch_username": os.getenv("OPENSEARCH_USERNAME", "admin"),
    "opensearch_password": os.getenv("OPENSEARCH_PASSWORD"),
    "index_name": "rag-support-demo-index",
}

QUESTIONS = [
    "What compensation is offered when a package arrives damaged?",
    "What happens after an account cancellation is processed?",
    "How can a customer update the credit card on their account?",
    "Where do you go in the interface to manage payment options?",
]

EXPECTED_ANSWERS = {
    "What compensation is offered when a package arrives damaged?": [
        "refund",
        "gift card",
    ],
    "What happens after an account cancellation is processed?": [
        "canceled",
        "refund",
    ],
    "How can a customer update the credit card on their account?": [
        "my account",
        "payment options",
        "add card info",
        "save",
    ],
    "Where do you go in the interface to manage payment options?": [
        "my account",
        "payment options",
    ],
}

print("Selected corpus files:")
for path in CONFIG["documents"]:
    print(f"  - {path}")

print("\nFixed evaluation questions:")
for i, question in enumerate(QUESTIONS, 1):
    print(f"  {i}. {question}")

In [ ]:
def check_services():
    services_ok = True

    print("Checking selected corpus files...")
    for path_str in CONFIG["documents"]:
        path = Path(path_str)
        if path.exists():
            print(f"  OK: {path}")
        else:
            print(f"  MISSING: {path}")
            services_ok = False

    print("\nChecking Ollama...")
    try:
        response = requests.get(f"{CONFIG['ollama_base']}/api/tags", timeout=3)
        response.raise_for_status()
        models = [m["name"] for m in response.json().get("models", [])]
        print("  Ollama is reachable")
        if not any("nomic-embed-text" in name for name in models):
            print("  Missing embedding model: nomic-embed-text")
            services_ok = False
        if not any(CONFIG["llm_model"] in name for name in models):
            print(f"  Missing generation model: {CONFIG['llm_model']}")
            services_ok = False
    except Exception as exc:
        print(f"  Ollama check failed: {exc}")
        services_ok = False

    print("\nChecking OpenSearch...")
    try:
        _scheme = 'https' if CONFIG['opensearch_use_ssl'] else 'http'
        response = requests.get(
            f"{_scheme}://{CONFIG['opensearch_host']}:{CONFIG['opensearch_port']}",
            auth=(CONFIG["opensearch_username"], CONFIG["opensearch_password"]),
            verify=False,
            timeout=5,
        )
        if response.status_code == 200:
            version = response.json().get("version", {}).get("number", "unknown")
            print(f"  OpenSearch is reachable (version {version})")
        else:
            print(f"  OpenSearch responded with status {response.status_code}")
            services_ok = False
    except requests.exceptions.ConnectionError:
        print("  OpenSearch not running (connection refused)")
        services_ok = False
    except requests.exceptions.Timeout:
        print("  OpenSearch not responding (timeout)")
        services_ok = False
    except Exception as exc:
        print(f"  OpenSearch check failed: {type(exc).__name__}: {exc}")
        services_ok = False

    return services_ok


if check_services():
    print("\nAll prerequisites look ready.")
else:
    print("\nFix the missing prerequisites before running the indexing flow.")

## Helper Functions

These helpers reuse the parquet-loading pattern from [`03_document_extraction.ipynb`](examples/notebooks/03_document_extraction.ipynb) so we can inspect intermediate outputs after each operator.

In [ ]:
def shorten_doc_name(value):
    if value is None:
        return None
    return Path(str(value)).name


def load_operator_results(manager, operator_name, operator_index=0):
    import pyarrow.parquet as pq

    try:
        session_info = manager.session_info
        job_id = session_info.job_id
        job_run_id = session_info.job_run_id
        operator_dir = f"{operator_name}_{operator_index}"
        result_file = Path(f"./data/{job_id}/{job_run_id}/data/{operator_dir}/output.parquet")

        if result_file.exists():
            result_table = pq.read_table(result_file)
            print(f"Loaded {operator_name} results from: {result_file}")
            print(f"Rows: {len(result_table)}")
            print(f"Columns: {result_table.column_names}")
            return result_table

        print(f"Result file not found: {result_file}")
        return None
    except Exception as exc:
        print(f"Error loading {operator_name} results: {exc}")
        print("Make sure 'data_storage_type': 'local' is set in global_config")
        return None


def display_table_preview(table, title, columns=None, max_rows=5, max_col_width=120):
    if table is None or len(table) == 0:
        print(f"No rows available for {title}")
        return

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    print(f"Rows: {len(table)}")
    print(f"Columns: {table.column_names}")

    df = table.to_pandas()
    if "name" in df.columns:
        df["name"] = df["name"].apply(shorten_doc_name)
    if "doc_name" in df.columns:
        df["doc_name"] = df["doc_name"].apply(shorten_doc_name)

    if columns:
        existing_columns = [col for col in columns if col in df.columns]
        if existing_columns:
            df = df[existing_columns]

    preview_df = df.head(max_rows).copy()
    for col in preview_df.columns:
        preview_df[col] = preview_df[col].apply(
            lambda value: str(value)[:max_col_width] + ("..." if len(str(value)) > max_col_width else "")
        )

    display(preview_df)


def display_chunk_preview(table, max_docs=3, max_chunks_per_doc=2, max_chunk_chars=180):
    if table is None or len(table) == 0 or "chunked_content" not in table.column_names:
        print("No chunked_content available")
        return

    print("\n" + "=" * 80)
    print("CHUNK PREVIEW")
    print("=" * 80)

    for row_index in range(min(len(table), max_docs)):
        row = table.slice(row_index, 1).to_pylist()[0]
        doc_name = shorten_doc_name(row.get("name", "unknown"))
        chunks = row.get("chunked_content") or []
        print(f"\nDocument: {doc_name}")
        print(f"Total chunks: {len(chunks)}")
        for chunk_index, chunk in enumerate(chunks[:max_chunks_per_doc], 1):
            chunk_text = chunk.get("chunk", "")
            preview = chunk_text[:max_chunk_chars].replace("\n", " ")
            print(f"  Chunk {chunk_index}: {preview}{'...' if len(chunk_text) > max_chunk_chars else ''}")


def display_embeddings_summary(table, max_rows=5):
    if table is None or len(table) == 0:
        print("No embeddings output available")
        return

    print("\n" + "=" * 80)
    print("EMBEDDINGS SUMMARY")
    print("=" * 80)
    print(f"Rows: {len(table)}")

    df = table.to_pandas()
    summary_rows = []
    for _, row in df.head(max_rows).iterrows():
        embedding_dimension = None
        embedding_preview = None
        embedding_storage = None

        embedding_value = row.get("embeddings")
        if isinstance(embedding_value, dict):
            embedding_storage = "memmap_file"
            embedding_preview = str(embedding_value)
        elif hasattr(embedding_value, "tolist"):
            embedding_list = embedding_value.tolist()
            if isinstance(embedding_list, list) and embedding_list:
                embedding_storage = "in_memory_array"
                if isinstance(embedding_list[0], list):
                    embedding_dimension = len(embedding_list[0]) if embedding_list[0] else 0
                    embedding_preview = embedding_list[0][:5]
                else:
                    embedding_dimension = len(embedding_list)
                    embedding_preview = embedding_list[:5]
        elif isinstance(embedding_value, list) and embedding_value:
            embedding_storage = "in_memory_list"
            if isinstance(embedding_value[0], list):
                embedding_dimension = len(embedding_value[0]) if embedding_value[0] else 0
                embedding_preview = embedding_value[0][:5]
            else:
                embedding_dimension = len(embedding_value)
                embedding_preview = embedding_value[:5]

        summary_rows.append(
            {
                "name": shorten_doc_name(row.get("name")),
                "doc_id_hash": row.get("doc_id_hash"),
                "embedding_storage": embedding_storage,
                "embedding_dimension": embedding_dimension,
                "embedding_preview": embedding_preview,
            }
        )

    display(pd.DataFrame(summary_rows))


def cleanup_flow_data(manager):
    try:
        session_info = manager.session_info
        job_id = session_info.job_id
        data_dir = Path(f"./data/{job_id}")
        if data_dir.exists():
            shutil.rmtree(data_dir)
            print(f"Cleaned up local flow data: {data_dir}")
        else:
            print(f"No local flow data found at: {data_dir}")
    except Exception as exc:
        print(f"Cleanup failed: {exc}")


print("Helper functions loaded successfully")

## Build the Indexing Flow

This notebook uses a **single indexing flow**. Retrieval and generation happen afterward in regular notebook cells.

In [ ]:
rag_indexing_flow = {
    "flow_name": "rag-support-demo",
    "description": "Index three support documents for retrieval-augmented QA",
    "global_config": {
        "doc_column": "content",
        "disable_validation": False,
        "force_ingest": True,
        "data_storage_type": "local"
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": [CONFIG["documents"]]},
                "include_filter": "txt"

            }}
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library",
                    "doc_column": "content"
                },
                "entity_extraction": {
                    "provider": "none"
                }
            }
        },
        {
            "name": "chunk",
            "type": "chunker",
            "depends_on": ["extract"],
            "config": {
                "chunk_type": "simple",
                "chunk_size": CONFIG["chunk_size"],
                "chunk_overlap": CONFIG["chunk_overlap"],
                "doc_column": "content",
                "retain_original_content": False
            }
        },
        {
            "name": "embeddings",
            "type": "embeddings",
            "depends_on": ["chunk"],
            "config": {
                "provider": "litellm",
                "embeddings_column": "embeddings",
                "doc_column": "content",
                "provider_config": {
                    "model_id": CONFIG["embedding_model"],
                    "api_base": CONFIG["ollama_base"],
                    "api_key": "not-required-for-local-ollama"
                }
            }
        },
        {
            "name": "vectordb",
            "type": "vectordb",
            "depends_on": ["embeddings"],
            "config": {
                "provider": "opensearch",
                "index_name": CONFIG["index_name"],
                "doc_id_column": "doc_id_hash",
                "embeddings_column": "embeddings",
                "vector_dimension": CONFIG["embedding_dimension"],
                "create_index": True,
                "provider_config": {
                    "host": CONFIG["opensearch_host"],
                    "port": CONFIG["opensearch_port"],
                    "username": CONFIG["opensearch_username"],
                    "password": CONFIG["opensearch_password"],
                    "use_ssl": CONFIG["opensearch_use_ssl"],
                    "verify_certs": False,
                    "engine": "faiss",
                    "algorithm": "hnsw",
                    "space_type": "l2",
                    "batch_size": 100
                },
                "available_features": {
                    "doc_id_hash": {
                        "name": "Document ID",
                        "available_for_vector_db": True,
                        "mandatory_for_vector_db": True,
                        "type": "string",
                        "is_primary": True
                    },
                    "content": {
                        "name": "Content",
                        "available_for_vector_db": True,
                        "type": "string"
                    },
                    "name": {
                        "name": "Document Name",
                        "available_for_vector_db": True,
                        "type": "string"
                    },
                    "path": {
                        "name": "File Path",
                        "available_for_vector_db": True,
                        "type": "string"
                    },
                    "embeddings": {
                        "name": "Embeddings",
                        "available_for_vector_db": True,
                        "mandatory_for_vector_db": True,
                        "type": "vector"
                    }
                },
                "feature_mappings": {
                    "doc_id_hash": "pk",
                    "content": "text",
                    "name": "doc_name",
                    "path": "file_path",
                    "embeddings": "vector_embeddings"
                }
            }
        }
    ]
}

print("Indexing flow ready.")
print("Stages: ingest -> extract -> chunk -> embeddings -> vectordb")
print(f"Index name: {CONFIG['index_name']}")

## Run the Indexing Flow

In [ ]:
print("Running indexing flow...")
manager = DocpipeFlowManager(flow_def=rag_indexing_flow)
manager.execute()
print("\nIndexing flow completed.")
print(f"Job ID: {manager.session_info.job_id}")
print(f"Job Run ID: {manager.session_info.job_run_id}")

## Inspect Intermediate Outputs

These previews help verify what each stage produced before we move on to retrieval and generation.

In [ ]:
ingest_table = load_operator_results(manager, operator_name="ingest", operator_index=0)
display_table_preview(
    ingest_table,
    title="Ingest Output Preview",
    columns=["name", "document_format", "size", "modified_time"],
    max_rows=5,
)

In [ ]:
extract_table = load_operator_results(manager, operator_name="extract", operator_index=0)
display_table_preview(
    extract_table,
    title="Extract Output Preview",
    columns=["name", "content"],
    max_rows=3,
    max_col_width=300,
)

In [ ]:
chunk_table = load_operator_results(manager, operator_name="chunk", operator_index=0)
display_table_preview(
    chunk_table,
    title="Chunk Output Preview",
    columns=["name", "chunked_content"],
    max_rows=5,
    max_col_width=220,
)
display_chunk_preview(chunk_table)

In [ ]:
embeddings_table = load_operator_results(manager, operator_name="embeddings", operator_index=0)
display_embeddings_summary(embeddings_table)

## Inspect the OpenSearch Index

Before asking questions, verify what was stored in the vector index.

In [ ]:
def opensearch_get(path, method="GET", body=None):
    _scheme = 'https' if CONFIG['opensearch_use_ssl'] else 'http'
    url = f"{_scheme}://{CONFIG['opensearch_host']}:{CONFIG['opensearch_port']}/{path.lstrip('/')}"
    response = requests.request(
        method=method,
        url=url,
        auth=(CONFIG["opensearch_username"], CONFIG["opensearch_password"]),
        headers={"Content-Type": "application/json"},
        data=json.dumps(body) if body is not None else None,
        verify=False,
        timeout=10,
    )
    response.raise_for_status()
    return response.json()


stats = opensearch_get(f"{CONFIG['index_name']}/_stats")
mapping = opensearch_get(f"{CONFIG['index_name']}/_mapping")
sample_docs = opensearch_get(
    f"{CONFIG['index_name']}/_search",
    method="POST",
    body={
        "size": 5,
        "query": {"match_all": {}},
        "_source": {"excludes": ["vector_embeddings"]},
    },
)

doc_count = stats["_all"]["primaries"]["docs"]["count"]
size_mb = stats["_all"]["primaries"]["store"]["size_in_bytes"] / 1024 / 1024
properties = mapping[CONFIG["index_name"]]["mappings"]["properties"]

print(f"Index: {CONFIG['index_name']}")
print(f"Document count: {doc_count}")
print(f"Index size: {size_mb:.2f} MB")
print("\nFields:")
for field_name, field_config in properties.items():
    field_type = field_config.get("type", "unknown")
    if field_type == "knn_vector":
        print(f"  - {field_name}: {field_type} (dimension={field_config.get('dimension')})")
    else:
        print(f"  - {field_name}: {field_type}")

hits = sample_docs["hits"]["hits"]
sample_rows = []
for hit in hits:
    source = hit.get("_source", {})
    sample_rows.append(
        {
            "doc_name": shorten_doc_name(source.get("doc_name") or source.get("name")),
            "opensearch_id": str(hit.get("_id", ""))[:12],
            "text_preview": str(source.get("text", ""))[:180],
        }
    )

display(pd.DataFrame(sample_rows))

## Run Real Retrieval

Retrieval is used here to gather evidence for answer generation.

In [ ]:
def get_query_embedding(query_text):
    response = requests.post(
        f"{CONFIG['ollama_base']}/api/embeddings",
        json={"model": "nomic-embed-text", "prompt": query_text},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["embedding"]


def retrieve_chunks(query_text, k=3):
    query_embedding = get_query_embedding(query_text)
    body = {
        "size": k,
        "query": {
            "knn": {
                "vector_embeddings": {
                    "vector": query_embedding,
                    "k": k,
                }
            }
        },
        "_source": ["text", "doc_name", "name", "file_path"],
    }
    results = opensearch_get(f"{CONFIG['index_name']}/_search", method="POST", body=body)
    return results["hits"]["hits"]


def display_retrieval_results(query_text, hits):
    print("\n" + "=" * 80)
    print(f"QUESTION: {query_text}")
    print("=" * 80)
    rows = []
    for rank, hit in enumerate(hits, 1):
        source = hit.get("_source", {})
        rows.append(
            {
                "rank": rank,
                "score": round(hit.get("_score", 0.0), 4),
                "doc_name": shorten_doc_name(source.get("doc_name") or source.get("name")),
                "text_preview": str(source.get("text", ""))[:220],
            }
        )
    display(pd.DataFrame(rows))

In [ ]:
retrieval_results = {}
for question in QUESTIONS:
    hits = retrieve_chunks(question, k=3)
    retrieval_results[question] = hits
    display_retrieval_results(question, hits)

## Run Real Generation

Now we build a grounded prompt from the retrieved chunks and ask Ollama to answer using only that evidence.

In [ ]:
def build_grounded_prompt(question, hits):
    context_blocks = []
    for rank, hit in enumerate(hits, 1):
        source = hit.get("_source", {})
        doc_name = shorten_doc_name(source.get("doc_name") or source.get("name") or "unknown")
        text = source.get("text", "")
        context_blocks.append(
            f"[Source {rank}] {doc_name}\n{text}"
        )

    context = "\n\n".join(context_blocks)
    return f"""You are answering a user question using retrieved support documents.

Use only the provided context.
If the answer is not supported by the context, say so clearly.
Answer directly when the context clearly states the answer.
Cite the source numbers you used.

Question:
{question}

Context:
{context}

Answer:
"""


def generate_grounded_answer(question, hits):
    prompt = build_grounded_prompt(question, hits)
    response = requests.post(
        f"{CONFIG['ollama_base']}/api/generate",
        json={
            "model": CONFIG["llm_model"],
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.0},
        },
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["response"], prompt


def display_generation_result(question, hits, answer):
    print("\n" + "=" * 80)
    print(f"QUESTION: {question}")
    print("=" * 80)
    print("Retrieved sources:")
    for rank, hit in enumerate(hits, 1):
        source = hit.get("_source", {})
        doc_name = shorten_doc_name(source.get('doc_name') or source.get('name'))
        text_preview = str(source.get('text', '')).replace("\n", " ")[:120]
        suffix = '...' if len(str(source.get('text', ''))) > 120 else ''
        print(f"  [Source {rank}] {doc_name}: {text_preview}{suffix}")
    print("\nGenerated answer:")
    print(answer.strip())

In [ ]:
generated_answers = {}
for question in QUESTIONS:
    hits = retrieval_results[question]
    answer, prompt = generate_grounded_answer(question, hits)
    generated_answers[question] = {
        "answer": answer,
        "prompt": prompt,
        "hits": hits,
    }
    display_generation_result(question, hits, answer)

## Evaluate the Answers

The notebook now includes a lightweight correctness check.

This is not a full semantic evaluator. It checks whether each generated answer contains expected key phrases derived from the source documents. That gives a quick signal about whether the answer is roughly correct and grounded.

In [ ]:
def evaluate_answer_contains_expected_terms(answer, expected_terms):
    normalized_answer = answer.lower()
    matched_terms = [term for term in expected_terms if term.lower() in normalized_answer]
    minimum_required = max(1, len(expected_terms) - 1)
    if len(expected_terms) <= 3:
        minimum_required = max(1, len(expected_terms) - 1)
    return {
        "matched_terms": matched_terms,
        "missing_terms": [term for term in expected_terms if term not in matched_terms],
        "match_count": len(matched_terms),
        "expected_count": len(expected_terms),
        "passed": len(matched_terms) >= minimum_required,
    }


evaluation_rows = []
for question in QUESTIONS:
    answer = generated_answers[question]["answer"]
    evaluation = evaluate_answer_contains_expected_terms(answer, EXPECTED_ANSWERS[question])
    evaluation_rows.append(
        {
            "question": question,
            "passed": evaluation["passed"],
            "matched_terms": ", ".join(evaluation["matched_terms"]),
            "missing_terms": ", ".join(evaluation["missing_terms"]),
            "match_count": f"{evaluation['match_count']}/{evaluation['expected_count']}",
        }
    )

display(pd.DataFrame(evaluation_rows))

## What to Look For

When reviewing the results, check:

1. Did retrieval return the right support document?
2. Did the answer stay grounded in the retrieved text?
3. Did the answer cite the relevant sources?
4. Did the lightweight evaluation agree with your manual judgment?
5. If retrieval was weak, did generation avoid inventing unsupported details?

The evaluation cell above gives a quick automated signal, but manual inspection still matters.

## Cleanup

The next cell automatically removes the local parquet artifacts created for stage inspection.

In [ ]:
cleanup_flow_data(manager)

## Summary

In this notebook you:

1. Indexed a small support-document corpus with a single docling-pipelines flow
2. Inspected intermediate outputs after ingest, extract, chunk, and embeddings
3. Verified what was stored in OpenSearch
4. Retrieved relevant chunks for fixed user questions
5. Generated grounded answers with Ollama using retrieved evidence
6. Ran a lightweight correctness check against expected answer terms
7. Automatically cleaned up local parquet artifacts after the notebook finished

## Related Notebooks

- [`04_embeddings_vectordb.ipynb`](examples/notebooks/04_embeddings_vectordb.ipynb) - embeddings, vector storage, and similarity search mechanics
- [`07_flow_authoring.ipynb`](examples/notebooks/07_flow_authoring.ipynb) - programmatic flow creation